# JSON Transformations

## Overview
This notebook covers building JSON output from relational data, aggregating rows into JSON arrays, and updating JSON values in place.

**Building JSON in SQL:**

| Need | SQLite | PostgreSQL |
|---|---|---|
| Object from columns | `json_object(k,v,k,v)` | `json_build_object(k,v,k,v)` |
| Array from values | `json_array(v1,v2)` | `json_build_array(v1,v2)` |
| Aggregate rows to array | `json_group_array(expr)` | `json_agg(expr)` |
| Aggregate to object | `json_group_object(k,v)` | `json_object_agg(k,v)` |
| Entire row as JSON | No direct equivalent | `row_to_json(t.*)` |

**Updating JSON values:**

| Need | SQLite | PostgreSQL |
|---|---|---|
| Set a path | `json_set(col, '$.key', val)` | `jsonb_set(col, '{key}', val)` |
| Remove a key | `json_remove(col, '$.key')` | `col - 'key'` |
| Merge objects | `json_patch(col, patch)` | `col \|\| patch::jsonb` |

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== Building JSON from relational rows ===")
print()

# json_object — build an object from key-value pairs
rows = conn.execute("""
    SELECT json_object(
        'patient_id', patient_id,
        'name',       full_name,
        'province',   json_extract(demographics,'$.province'),
        'age',        json_extract(demographics,'$.age')
    ) AS patient_summary
    FROM patients
    ORDER BY full_name
    LIMIT 3
""").fetchall()
print("json_object: build JSON from columns:")
for r in rows:
    print(f"  {r['patient_summary']}")

print()
# json_group_array — aggregate rows into a JSON array
rows2 = conn.execute("""
    SELECT json_extract(demographics,'$.province') AS province,
           json_group_array(
               json_object('id', patient_id, 'name', full_name)
           ) AS patients_json
    FROM   patients
    GROUP  BY province
    ORDER  BY province
""").fetchall()
print("json_group_array: aggregate into JSON array per province:")
for r in rows2:
    arr = json.loads(r['patients_json'])
    print(f"  {r['province']}: {arr}")

print()
print("PostgreSQL equivalents:")
pg_build = [
    ("json_object(k1,v1,k2,v2)",    "json_build_object(k1,v1,k2,v2) OR row_to_json(row)"),
    ("json_group_array(...)",        "json_agg(expr)           -- aggregate rows to JSON array"),
    ("json_group_object(k,v)",       "json_object_agg(k,v)    -- aggregate to JSON object"),
    ("json_array(v1,v2,v3)",        "json_build_array(v1,v2,v3)"),
    ("No direct equivalent",        "to_json(val)            -- any PostgreSQL value to JSON"),
    ("No direct equivalent",        "row_to_json(t)          -- entire row to JSON"),
]
for sqlite, pg in pg_build:
    print(f"  {sqlite:<38s}  {pg}")


Healthcare JSON dataset ready:
  patients: 5 rows
  lab_results: 4 rows
  medications: 6 rows

Sample demographics JSON for Alice Ngata:
{
  "age": 45,
  "dob": "1980-03-15",
  "province": "NB",
  "contact": {
    "phone": "506-555-0101",
    "email": "alice@email.com"
  },
  "insurance": {
    "plan": "premium",
    "id": "INS-00123"
  }
}
=== Building JSON from relational rows ===

json_object: build JSON from columns:
  {"patient_id":"P001","name":"Alice Ngata","province":"NB","age":45}
  {"patient_id":"P002","name":"Bob Chen","province":"ON","age":52}
  {"patient_id":"P003","name":"Fatima Rashid","province":"BC","age":38}

json_group_array: aggregate into JSON array per province:
  BC: [{'id': 'P003', 'name': 'Fatima Rashid'}]
  NB: [{'id': 'P001', 'name': 'Alice Ngata'}]
  NS: [{'id': 'P004', 'name': 'James MacLeod'}]
  ON: [{'id': 'P002', 'name': 'Bob Chen'}]
  QC: [{'id': 'P005', 'name': 'Mei-Ling Tran'}]

PostgreSQL equivalents:
  json_object(k1,v1,k2,v2)                json_bu

---
## json_agg and row_to_json patterns

In [2]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== json_agg and row_to_json (PostgreSQL patterns) ===")
print()

# Simulate json_agg pattern: medications per patient as nested JSON
rows = conn.execute("""
    SELECT p.patient_id,
           p.full_name,
           json_group_array(
               json_object(
                   'drug',      json_extract(m.details,'$.drug'),
                   'dose',      json_extract(m.details,'$.dose'),
                   'frequency', json_extract(m.details,'$.frequency')
               )
           ) AS medications
    FROM   patients p
    LEFT JOIN medications m ON m.patient_id = p.patient_id
    GROUP  BY p.patient_id, p.full_name
    ORDER  BY p.full_name
""").fetchall()

print("Patient with nested medications (json_group_array):")
for r in rows:
    meds = json.loads(r['medications'])
    print(f"  {r['full_name']:<22s}  medications: {meds}")

print()
print("PostgreSQL json_agg + row_to_json patterns:")
pg_agg = """
-- json_agg: rows to JSON array
SELECT p.full_name,
       json_agg(json_build_object(
           'drug',      d.details->>'drug',
           'dose',      d.details->>'dose',
           'frequency', d.details->>'frequency'
       )) AS medications
FROM   patients p
LEFT JOIN medications d ON d.patient_id = p.patient_id
GROUP  BY p.full_name
ORDER  BY p.full_name;

-- row_to_json: entire row as JSON
SELECT row_to_json(p.*) FROM patients p LIMIT 1;

-- json_object_agg: aggregate to JSON object (key->value)
SELECT json_object_agg(patient_id, full_name)
FROM   patients;
-- Returns: {"P001": "Alice Ngata", "P002": "Bob Chen", ...}

-- to_jsonb: convert a value or row to JSONB
SELECT to_jsonb(ARRAY[1,2,3]);         -- [1, 2, 3]
SELECT to_jsonb(ROW('Alice', 45));     -- {"f1":"Alice","f2":45}
"""
print(pg_agg)


Healthcare JSON dataset ready:
  patients: 5 rows
  lab_results: 4 rows
  medications: 6 rows

Sample demographics JSON for Alice Ngata:
{
  "age": 45,
  "dob": "1980-03-15",
  "province": "NB",
  "contact": {
    "phone": "506-555-0101",
    "email": "alice@email.com"
  },
  "insurance": {
    "plan": "premium",
    "id": "INS-00123"
  }
}
=== json_agg and row_to_json (PostgreSQL patterns) ===

Patient with nested medications (json_group_array):
  Alice Ngata             medications: [{'drug': 'Atorvastatin', 'dose': '40mg', 'frequency': 'once daily'}, {'drug': 'Lisinopril', 'dose': '10mg', 'frequency': 'once daily'}]
  Bob Chen                medications: [{'drug': 'Lisinopril', 'dose': '5mg', 'frequency': 'once daily'}, {'drug': 'Metformin', 'dose': '500mg', 'frequency': 'BID'}]
  Fatima Rashid           medications: [{'drug': 'Salbutamol', 'dose': '2.5mg', 'frequency': 'PRN'}]
  James MacLeod           medications: [{'drug': None, 'dose': None, 'frequency': None}]
  Mei-Ling Tran  

---
## Updating JSON: json_set and jsonb_set

In [3]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== Updating JSON data: json_patch and jsonb_set ===")
print()

# SQLite: update a JSON value using json_set / json_patch
print("SQLite json_set — update a specific path:")
rows_before = conn.execute("SELECT patient_id, json_extract(demographics,'$.age') AS age FROM patients WHERE patient_id='P001'").fetchone()
print(f"  P001 age before: {rows_before['age']}")

conn.execute("""
    UPDATE patients
    SET    demographics = json_set(demographics, '$.age', 46)
    WHERE  patient_id = 'P001'
""")
conn.commit()
rows_after = conn.execute("SELECT patient_id, json_extract(demographics,'$.age') AS age FROM patients WHERE patient_id='P001'").fetchone()
print(f"  P001 age after:  {rows_after['age']}")

print()
# json_insert (only if path doesn't exist), json_remove
conn.execute("""
    UPDATE patients
    SET    preferences = json_set(COALESCE(preferences,'{}'), '$.last_login', '2026-03-11')
    WHERE  patient_id = 'P001'
""")
conn.commit()
prefs = conn.execute("SELECT json_extract(preferences,'$.last_login') AS ll FROM patients WHERE patient_id='P001'").fetchone()
print(f"  P001 preferences.last_login set to: {prefs['ll']}")

print()
print("SQLite JSON modification functions:")
sqlite_fns = [
    ("json_set(json, path, val)",    "Inserts or replaces at path"),
    ("json_insert(json, path, val)", "Inserts at path only if it doesn't exist"),
    ("json_replace(json, path, val)","Replaces at path only if it exists"),
    ("json_remove(json, path)",      "Removes the value at path"),
    ("json_patch(json, patch)",      "Applies RFC 7396 JSON Merge Patch"),
]
for fn, desc in sqlite_fns:
    print(f"  {fn:<36s}  {desc}")

print()
print("PostgreSQL JSONB update functions:")
pg_fns = [
    ("jsonb_set(col, '{key}', val)",          "Set value at path (creates path if needed)"),
    ("jsonb_set(col, '{a,b}', val)",          "Set nested path"),
    ("jsonb_insert(col, '{key}', val)",       "Insert at path (does not overwrite existing)"),
    ("col - 'key'",                           "Remove top-level key"),
    ("col #- '{a,b}'",                        "Remove nested key"),
    ("col || '{\"key\":\"val\"}'::jsonb",      "Merge / overwrite with new JSONB"),
    ("jsonb_set(col, '{arr,0}', val)",        "Update first array element"),
]
for fn, desc in pg_fns:
    print(f"  {fn:<48s}  {desc}")


Healthcare JSON dataset ready:
  patients: 5 rows
  lab_results: 4 rows
  medications: 6 rows

Sample demographics JSON for Alice Ngata:
{
  "age": 45,
  "dob": "1980-03-15",
  "province": "NB",
  "contact": {
    "phone": "506-555-0101",
    "email": "alice@email.com"
  },
  "insurance": {
    "plan": "premium",
    "id": "INS-00123"
  }
}
=== Updating JSON data: json_patch and jsonb_set ===

SQLite json_set — update a specific path:
  P001 age before: 45
  P001 age after:  46

  P001 preferences.last_login set to: 2026-03-11

SQLite JSON modification functions:
  json_set(json, path, val)             Inserts or replaces at path
  json_insert(json, path, val)          Inserts at path only if it doesn't exist
  json_replace(json, path, val)         Replaces at path only if it exists
  json_remove(json, path)               Removes the value at path
  json_patch(json, patch)               Applies RFC 7396 JSON Merge Patch

PostgreSQL JSONB update functions:
  jsonb_set(col, '{key}', val)

---
## Common Pitfalls

**1. json_agg including NULLs from LEFT JOIN**
`json_agg(m.details)` on a LEFT JOIN returns `[null]` for patients with no medications instead of `[]`. Use `json_agg(m.details) FILTER (WHERE m.details IS NOT NULL)` or `COALESCE(json_agg(m.details) FILTER (WHERE m.med_id IS NOT NULL), '[]'::json)` to return an empty array.

**2. row_to_json exposing sensitive fields**
`SELECT row_to_json(p.*) FROM patients p` includes ALL columns, including PII and internal fields. Never use `row_to_json(t.*)` for API responses — build the JSON explicitly with `json_build_object('name', p.name, 'province', p.province)`.

**3. jsonb_set not creating intermediate paths**
`jsonb_set(col, '{new,nested,key}', value)` fails if `new` or `nested` don't already exist in the JSONB. Use `jsonb_set(col, '{key}', value, true)` (the 4th argument `create_missing=true`) for top-level keys. For deep paths, create parent objects first.

**4. json_group_array ordering in SQLite**
`json_group_array(expr)` in SQLite does not guarantee order unless you include an `ORDER BY` in the query. For ordered arrays (e.g., vitals sorted by date), always add `ORDER BY date` before the GROUP BY.

**5. Performance of row_to_json on wide tables**
`row_to_json` on a table with 50+ columns serialises all columns including large TEXT fields. For API responses, project only the needed columns into a subquery or CTE before calling `row_to_json`.

**6. Modifying JSONB inside a CTE without RETURNING**
A CTE with `UPDATE ... SET col = jsonb_set(col, ...) RETURNING col` is the correct pattern for getting the post-update JSONB value. Without `RETURNING`, you need a second query to read the updated value, which adds a round-trip and a race condition.


---
*sql_methods_library - Samantha McGarrigle*